# Final Test Results

Runs test set with optuna tuned HPs for all 21 **model x representation** combinations. These are the results reported in Table 1.

In [1]:
import numpy as np
import pandas as pd
import json, os, random, pickle
from copy import deepcopy
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout, BatchNorm1d, Sequential, ReLU
from torch_geometric.nn import GCNConv, GINEConv, GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader
from torch.optim import Adam

from scipy.special import expit
from scipy.optimize import minimize_scalar

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, precision_score,
    recall_score, classification_report, roc_curve
)
from xgboost import XGBClassifier

import sys
from pathlib import Path
sys.path.append(str(Path('/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/PBSG')))

import pbsg
from pbsg.polymer import polymers_from_df
from pbsg.polymer.generators import smiles_from_graphs
from pbsg.ml import to_pyg
from pbsg.ml.pyg import architecture_onehot
from pbsg.polymer.parser import parse_smiles_list
import ast

def safe_parse(val):
    try:
        if pd.isna(val): return None
    except (TypeError, ValueError): pass
    while isinstance(val, str):
        try: val = ast.literal_eval(val)
        except (ValueError, SyntaxError): break
    return val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

/Users/xgv7306/miniconda/envs/pbsg/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("best_hp_reg.json") as f:
    hp = json.load(f)
print(f"Entries: {len(hp)}")  # should be 21
print(sorted(hp.keys()))

Entries: 21
['gatv2_pbsg', 'gatv2_smiles', 'gatv2_smiles_gl', 'gcn_pbsg', 'gcn_smiles', 'gcn_smiles_gl', 'gine_pbsg', 'gine_smiles', 'gine_smiles_gl', 'rf_fp_poly', 'rf_fp_pooled_poly', 'rf_fp_pooled_ru', 'rf_fp_ru', 'rr_fp_poly', 'rr_fp_pooled_poly', 'rr_fp_pooled_ru', 'rr_fp_ru', 'xgb_fp_poly', 'xgb_fp_pooled_poly', 'xgb_fp_pooled_ru', 'xgb_fp_ru']


In [ ]:
df = pd.read_csv('data_regressor.csv')
for col in ['beads_list', 'lengths', 'reactivity_ratios', 'spatiality_rules']:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)
df['lengths']             = df['lengths'].apply(lambda x: None if isinstance(x, str) else x)
df['architecture_onehot'] = df.apply(architecture_onehot, axis=1)
df['beads_list']          = df['repeat_units'].apply(parse_smiles_list)

In [4]:
with open("random_stratified_splits_reg.json") as f:
    splits = json.load(f)

folds        = splits["folds"]
test_idx     = splits["test"]
all_idx      = np.arange(len(df))
trainval_idx = np.setdiff1d(all_idx, test_idx)
print(f"trainval={len(trainval_idx)}  test={len(test_idx)}  folds={len(folds)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

trainval=345  test=62  folds=5
Device: cpu


In [5]:
X_fp_RU        = np.load("Vectors/X_fp_RU.npy")
X_fp_pooled_RU    = np.load("Vectors/X_fp_pooled_RU.npy")
X_fp_poly      = np.load("Vectors/X_fp_poly.npy")
X_fp_pooled_poly = np.load("Vectors/X_fp_pooled_poly.npy")

graphs_PBSG = torch.load("Graphs/graphs_PBSG.pt")
graphs_SMILES = torch.load("Graphs/graphs_SMILES.pt")
graphs_SMILESplusglobal = torch.load("Graphs/graphs_SMILES+global.pt")

y=np.load("Vectors/y.npy")

In [8]:
import importlib
import scripts.test_eval_all_models_reg

# After you make a change to the .py file, run this:
importlib.reload(scripts.test_eval_all_models_reg)

<module 'scripts.test_eval_all_models_reg' from '/Users/xgv7306/Library/CloudStorage/OneDrive-NorthwesternUniversity/northwestern/2-PolyID:Dora/polymer_melting_temp_ML_clean/Regressor/scripts/test_eval_all_models_reg.py'>

In [9]:
# Inject y into graphs first
graph_reps = {
    "PBSG":          graphs_PBSG,
    "SMILES":        graphs_SMILES,
    "SMILES+global": graphs_SMILESplusglobal
}

for graphs in graph_reps.values():
    for i, g in enumerate(graphs):
        if g.y is None:
            import torch
            g.y = torch.tensor([y[i]], dtype=torch.float32)

In [10]:
from scripts.test_eval_all_models_reg import run_all_test_reg

test_results_reg = run_all_test_reg(
    graphs_PBSG, graphs_SMILES, graphs_SMILESplusglobal,
    X_fp_pooled_RU, X_fp_RU, X_fp_pooled_poly, X_fp_poly,
    y, df, trainval_idx, test_idx, device,
    hp_path="best_hp_reg.json",
)

Loaded HPs from best_hp_reg.json  (21 entries)

  Retraining GCN (PBSG) on full trainval (n=345)
  hidden=64, layers=2, dropout=0.496, lr=0.00299, seed=10
    Epoch  20 | train_loss=0.0928 val_loss=0.0898 val_mae=0.321 val_r2=0.820
    Epoch  40 | train_loss=0.0659 val_loss=0.0968 val_mae=0.339 val_r2=0.805
    Epoch  60 | train_loss=0.0576 val_loss=0.0814 val_mae=0.280 val_r2=0.834
    Epoch  80 | train_loss=0.0558 val_loss=0.0813 val_mae=0.278 val_r2=0.837
    Epoch 100 | train_loss=0.0618 val_loss=0.1150 val_mae=0.351 val_r2=0.764
    Early stop at epoch 117  best_val_r2=0.8748
  Training time: 0.14 min

  TEST RESULTS: GCN (PBSG)
  MAE                   : 16.3405
  RMSE                  : 23.2224
  R²                    : 0.8748
  MAPE (%)              : 13.8170
  Max Error             : 69.4525
  train_time_min        : 0.1400

  Subgroup: copolymer_type
  Group                    n      MAE     RMSE       R²
  -----------------------------------------------------
  alternating   

In [9]:
%run scripts/train_final_full_data.py

Full data: N=407  Tm range [-18,400] C

Training XGB full-data deployment model (xgb_fp_pooled_ru)...
  saved -> saved_models_final_fulldata/reg_xgb_pooled_ru_FULL.pkl

Done -> saved_models_final_fulldata/  (EXTERNAL predictions only)
